# 05 — Advanced Analytics
## Bluestock Mutual Fund Analytics Capstone — D6

**Objective:** Apply advanced financial analytics techniques beyond basic metrics.

Topics Covered
--------------
| # | Topic | Method |
|---|-------|--------|
| 1 | Value at Risk (VaR) | Historical + Parametric |
| 2 | Monte Carlo Simulation | GBM projection |
| 3 | Markowitz Efficient Frontier | Mean-Variance Optimisation |
| 4 | Fund Clustering | K-Means on metrics |
| 5 | Fund Recommendation Engine | Similarity scoring |
| 6 | Rolling CAGR Consistency | Trend analysis |


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import minimize
from scipy import stats as sp_stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

BASE    = Path().resolve().parents[1]
DB_PATH = BASE / "datasets" / "db"       / "bluestock_mf.db"
ANA_DIR = BASE / "datasets" / "analytics"

with sqlite3.connect(DB_PATH) as conn:
    nav_df = pd.read_sql_query("""
        SELECT nh.scheme_code, fm.scheme_name, cm.category, cm.sub_category,
               nh.date, nh.nav, nh.daily_return
        FROM nav_history nh
        JOIN fund_master fm ON nh.scheme_code=fm.scheme_code
        JOIN category_master cm ON fm.category_id=cm.category_id
        ORDER BY nh.scheme_code, nh.date
    """, conn, parse_dates=['date'])

    pm = pd.read_sql_query("""
        SELECT pm.*, fm.scheme_name, cm.category, cm.sub_category, fm.risk_level
        FROM performance_metrics pm
        JOIN fund_master fm ON pm.scheme_code=fm.scheme_code
        JOIN category_master cm ON fm.category_id=cm.category_id
    """, conn)

equity_nav = nav_df[nav_df['category']=='Equity'].copy()
returns_pivot = equity_nav.pivot_table(
    index='date', columns='scheme_name', values='daily_return'
).dropna(how='all')

print(f"Equity funds: {returns_pivot.shape[1]}")
print(f"Date range  : {returns_pivot.index.min().date()} → {returns_pivot.index.max().date()}")
print(f"Trading days: {len(returns_pivot)}")


In [ ]:
# ── 1. VALUE AT RISK (VaR) ────────────────────────────────────────────────────
CONFIDENCE_LEVELS = [0.95, 0.99]
INVESTMENT        = 1_000_000   # ₹10 Lakh

var_results = []
for col in returns_pivot.columns:
    rets = returns_pivot[col].dropna()
    for cl in CONFIDENCE_LEVELS:
        # Historical VaR
        hist_var  = np.percentile(rets, (1-cl)*100)
        # Parametric VaR (normal)
        param_var = rets.mean() + sp_stats.norm.ppf(1-cl) * rets.std()
        # CVaR (Expected Shortfall)
        cvar      = rets[rets <= hist_var].mean()
        var_results.append({
            'fund':              col[:30],
            'confidence':        f"{cl*100:.0f}%",
            'hist_var_pct':      round(hist_var*100, 4),
            'param_var_pct':     round(param_var*100, 4),
            'cvar_pct':          round(cvar*100, 4),
            'hist_var_inr':      round(hist_var * INVESTMENT, 0),
            'cvar_inr':          round(cvar * INVESTMENT, 0),
        })

var_df = pd.DataFrame(var_results)
var_df.to_csv(ANA_DIR / 'var_analysis.csv', index=False)

# Plot VaR comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
var_95 = var_df[var_df['confidence']=='95%'].sort_values('hist_var_pct')
var_99 = var_df[var_df['confidence']=='99%'].sort_values('hist_var_pct')

for ax, data, title in zip(axes, [var_95, var_99], ['95% VaR','99% VaR']):
    x = range(len(data))
    ax.bar([i-0.2 for i in x], data['hist_var_pct'],  0.4, label='Historical', color='steelblue', alpha=0.8)
    ax.bar([i+0.2 for i in x], data['param_var_pct'], 0.4, label='Parametric', color='darkorange', alpha=0.8)
    ax.plot(x, data['cvar_pct'], 'r^--', label='CVaR', lw=1.5, ms=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(data['fund'], rotation=35, ha='right', fontsize=8)
    ax.set_title(f'Daily {title} by Fund (%)', fontweight='bold')
    ax.set_ylabel('VaR %')
    ax.legend()

plt.tight_layout()
plt.savefig(ANA_DIR / 'adv_01_var.png', dpi=130, bbox_inches='tight')
plt.show()
print("VaR saved. Sample (95%):")
print(var_95[['fund','hist_var_pct','cvar_pct','hist_var_inr']].to_string(index=False))


In [ ]:
# ── 2. MONTE CARLO SIMULATION ─────────────────────────────────────────────────
N_SIMULATIONS = 1000
N_DAYS        = 252       # 1 year projection
INITIAL_NAV   = 100.0

# Pick top 3 equity funds by Sharpe
top_funds = pm[(pm['period_label']=='1Y') & (pm['category']=='Equity')]               .sort_values('sharpe_ratio', ascending=False).head(3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

mc_results = {}
for ax, (_, fund_row) in zip(axes, top_funds.iterrows()):
    name  = fund_row['scheme_name'][:30]
    rets  = returns_pivot[fund_row['scheme_name']].dropna() if fund_row['scheme_name'] in returns_pivot.columns else None
    if rets is None:
        ax.set_title(f'{name}\n(no data)')
        continue

    mu    = float(rets.mean())
    sigma = float(rets.std())

    # GBM simulation
    rng   = np.random.default_rng(42)
    Z     = rng.standard_normal((N_DAYS, N_SIMULATIONS))
    daily = np.exp((mu - 0.5*sigma**2) + sigma*Z)
    paths = INITIAL_NAV * np.cumprod(daily, axis=0)

    # Plot
    ax.plot(paths[:, :100], alpha=0.04, color='steelblue', lw=0.5)
    ax.plot(np.percentile(paths, 50, axis=1), color='navy',   lw=2, label='Median')
    ax.plot(np.percentile(paths, 5,  axis=1), color='red',    lw=1.5, ls='--', label='5th pct')
    ax.plot(np.percentile(paths, 95, axis=1), color='green',  lw=1.5, ls='--', label='95th pct')
    ax.axhline(INITIAL_NAV, color='black', lw=0.8, ls=':')

    final = paths[-1, :]
    mc_results[name] = {
        'median_final': round(np.median(final),2),
        'p5_final':     round(np.percentile(final, 5),2),
        'p95_final':    round(np.percentile(final, 95),2),
        'prob_profit':  round((final > INITIAL_NAV).mean() * 100, 1),
    }

    ax.set_title(f'MC: {name}\n{N_SIMULATIONS} paths, 1Y', fontweight='bold', fontsize=9)
    ax.set_ylabel('NAV')
    ax.set_xlabel('Trading Days')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ANA_DIR / 'adv_02_monte_carlo.png', dpi=130, bbox_inches='tight')
plt.show()

print("MONTE CARLO RESULTS (1Y projection, Base=100):")
for name, res in mc_results.items():
    print(f"  {name[:35]:35s} | Median:{res['median_final']:6.1f} "
          f"| 5th:{res['p5_final']:6.1f} | 95th:{res['p95_final']:6.1f} "
          f"| P(profit):{res['prob_profit']}%")


In [ ]:
# ── 3. MARKOWITZ EFFICIENT FRONTIER ───────────────────────────────────────────
# Use equity funds with full data
eq_rets = returns_pivot.dropna(axis=1, thresh=int(len(returns_pivot)*0.9))
eq_rets = eq_rets.dropna()
n_assets = eq_rets.shape[1]

mu_vec  = eq_rets.mean().values * 252       # annualised
cov_mat = eq_rets.cov().values  * 252       # annualised
rf      = 0.065

def portfolio_stats(weights):
    ret = weights @ mu_vec
    vol = np.sqrt(weights @ cov_mat @ weights)
    return ret, vol

def neg_sharpe(weights):
    ret, vol = portfolio_stats(weights)
    return -(ret - rf) / vol

constraints = [{'type':'eq','fun': lambda w: np.sum(w)-1}]
bounds = [(0, 1)] * n_assets

# Simulate random portfolios
np.random.seed(42)
n_portfolios = 3000
all_ret, all_vol, all_sr = [], [], []
for _ in range(n_portfolios):
    w  = np.random.dirichlet(np.ones(n_assets))
    r, v = portfolio_stats(w)
    all_ret.append(r); all_vol.append(v)
    all_sr.append((r - rf) / v if v > 0 else 0)

# Max Sharpe portfolio
init_w = np.ones(n_assets) / n_assets
res_sharpe = minimize(neg_sharpe, init_w, method='SLSQP',
                      bounds=bounds, constraints=constraints)
opt_w_sharpe = res_sharpe.x
opt_ret_s, opt_vol_s = portfolio_stats(opt_w_sharpe)

# Min Volatility portfolio
res_minvol = minimize(lambda w: portfolio_stats(w)[1], init_w,
                      method='SLSQP', bounds=bounds, constraints=constraints)
opt_w_minvol = res_minvol.x
opt_ret_m, opt_vol_m = portfolio_stats(opt_w_minvol)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))
sc = ax.scatter(all_vol, all_ret, c=all_sr, cmap='RdYlGn', s=5, alpha=0.4)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
ax.scatter(opt_vol_s, opt_ret_s, marker='*', s=400, c='gold',
           edgecolors='black', lw=1.5, zorder=5, label=f'Max Sharpe ({(opt_ret_s-rf)/opt_vol_s:.2f})')
ax.scatter(opt_vol_m, opt_ret_m, marker='D', s=200, c='cyan',
           edgecolors='black', lw=1.5, zorder=5, label='Min Volatility')

# Label individual funds
for i, name in enumerate(eq_rets.columns):
    ax.scatter(np.sqrt(cov_mat[i,i]), mu_vec[i],
               marker='o', s=80, c='navy', edgecolors='white', lw=0.5, zorder=4)
    ax.annotate(name[:15], (np.sqrt(cov_mat[i,i]), mu_vec[i]),
                fontsize=7, xytext=(4,4), textcoords='offset points')

ax.set_xlabel('Annualised Volatility'); ax.set_ylabel('Annualised Return')
ax.set_title('Markowitz Efficient Frontier — Equity Funds', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(ANA_DIR / 'adv_03_efficient_frontier.png', dpi=130, bbox_inches='tight')
plt.show()

print("OPTIMAL PORTFOLIOS:")
print(f"  Max Sharpe  : Return={opt_ret_s*100:.1f}%  Vol={opt_vol_s*100:.1f}%  Sharpe={(opt_ret_s-rf)/opt_vol_s:.3f}")
print(f"  Min Vol     : Return={opt_ret_m*100:.1f}%  Vol={opt_vol_m*100:.1f}%")
print()
print("MAX SHARPE PORTFOLIO WEIGHTS:")
for name, w in zip(eq_rets.columns, opt_w_sharpe):
    if w > 0.01:
        print(f"  {name[:35]:35s}: {w*100:.1f}%")

# Save weights
weights_df = pd.DataFrame({
    'fund': eq_rets.columns,
    'max_sharpe_weight': opt_w_sharpe.round(4),
    'min_vol_weight':    opt_w_minvol.round(4)
})
weights_df.to_csv(ANA_DIR / 'portfolio_weights.csv', index=False)


In [ ]:
# ── 4. FUND CLUSTERING ────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pm_1y_full = pm[pm['period_label']=='1Y'].dropna(
    subset=['cagr','volatility_ann','max_drawdown','sharpe_ratio','beta'])

features = ['cagr','volatility_ann','max_drawdown','sharpe_ratio','beta','alpha']
X = pm_1y_full[features].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow method
inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# Fit with k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
pm_1y_full = pm_1y_full.copy()
pm_1y_full['cluster'] = km.fit_predict(X_scaled)

cluster_labels = {0:'Conservative', 1:'Balanced', 2:'Aggressive', 3:'High-Alpha'}
pm_1y_full['cluster_name'] = pm_1y_full['cluster'].map(cluster_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(2,8), inertias, 'bo-', ms=8)
axes[0].set_title('Elbow Method — Optimal k', fontweight='bold')
axes[0].set_xlabel('Number of Clusters'); axes[0].set_ylabel('Inertia')

colors = ['steelblue','darkorange','green','red']
for cid, (cname, grp) in enumerate(pm_1y_full.groupby('cluster_name')):
    axes[1].scatter(grp['volatility_ann']*100, grp['cagr']*100,
                    c=colors[cid % 4], s=100, label=cname,
                    edgecolors='white', lw=0.5, alpha=0.85)
    for _, row in grp.iterrows():
        axes[1].annotate(row['scheme_name'][:12],
                         (row['volatility_ann']*100, row['cagr']*100),
                         fontsize=6, alpha=0.7,
                         xytext=(3,3), textcoords='offset points')

axes[1].set_title('Fund Clusters (K-Means, k=4)', fontweight='bold')
axes[1].set_xlabel('Volatility %'); axes[1].set_ylabel('CAGR %')
axes[1].legend()

plt.tight_layout()
plt.savefig(ANA_DIR / 'adv_04_clustering.png', dpi=130, bbox_inches='tight')
plt.show()

print("CLUSTER COMPOSITION:")
print(pm_1y_full.groupby('cluster_name')[['cagr','volatility_ann','sharpe_ratio']]
      .mean().round(4).to_string())
pm_1y_full[['scheme_name','cluster_name','cagr','volatility_ann','sharpe_ratio']].to_csv(
    ANA_DIR / 'fund_clusters.csv', index=False)


In [ ]:
# ── 5. FUND RECOMMENDATION ENGINE ─────────────────────────────────────────────
def recommend_funds(risk_appetite: str, investment_horizon: str,
                    top_n: int = 5) -> pd.DataFrame:
    """
    Simple rule-based + score-based fund recommender.

    Parameters
    ----------
    risk_appetite : 'low' | 'medium' | 'high'
    investment_horizon : 'short' | 'medium' | 'long'
    top_n : number of funds to return
    """
    # Map inputs to period
    horizon_map = {'short':'1Y', 'medium':'3Y', 'long':'5Y'}
    period = horizon_map.get(investment_horizon.lower(), '1Y')

    risk_map = {
        'low':    ['Low','Moderate'],
        'medium': ['Moderate','Moderately High'],
        'high':   ['Moderately High','High','Very High'],
    }
    allowed_risk = risk_map.get(risk_appetite.lower(), ['Moderate','Moderately High'])

    # Filter
    filtered = pm[(pm['period_label']==period) &
                  (pm['risk_level'].isin(allowed_risk))].copy()

    # Score = 40% CAGR + 40% Sharpe + 20% (1 - abs(MDD))
    filtered = filtered.dropna(subset=['cagr','sharpe_ratio','max_drawdown'])
    if len(filtered) == 0:
        return pd.DataFrame()

    # Normalise 0-1
    for col in ['cagr','sharpe_ratio']:
        mn, mx = filtered[col].min(), filtered[col].max()
        filtered[f'{col}_norm'] = (filtered[col]-mn) / (mx-mn+1e-9)
    filtered['mdd_norm'] = 1 - (filtered['max_drawdown'].abs() /
                                 filtered['max_drawdown'].abs().max())
    filtered['score'] = (0.40 * filtered['cagr_norm'] +
                         0.40 * filtered['sharpe_ratio_norm'] +
                         0.20 * filtered['mdd_norm'])

    result = filtered.sort_values('score', ascending=False).head(top_n)[[
        'scheme_name','sub_category','risk_level',
        'cagr','sharpe_ratio','max_drawdown','score'
    ]].copy()
    result['cagr']         = (result['cagr'] * 100).round(2)
    result['sharpe_ratio'] = result['sharpe_ratio'].round(3)
    result['max_drawdown'] = (result['max_drawdown'] * 100).round(2)
    result['score']        = result['score'].round(4)
    return result

# Demonstrate recommendations
print("=" * 65)
profiles = [
    ('low',    'short',  'Conservative Investor (Low Risk, Short Term)'),
    ('medium', 'medium', 'Balanced Investor (Medium Risk, Medium Term)'),
    ('high',   'long',   'Aggressive Investor (High Risk, Long Term)'),
]
for risk, horizon, label in profiles:
    recs = recommend_funds(risk, horizon)
    print(f"\n{label}:")
    print(recs[['scheme_name','sub_category','cagr','sharpe_ratio','score']].to_string(index=False))


In [ ]:
# ── 6. ROLLING CAGR CONSISTENCY ───────────────────────────────────────────────
# Rolling 1-year CAGR for each fund — shows consistency
with sqlite3.connect(DB_PATH) as conn:
    nav_all = pd.read_sql_query("""
        SELECT nh.scheme_code, fm.scheme_name, nh.date, nh.nav
        FROM nav_history nh JOIN fund_master fm ON nh.scheme_code=fm.scheme_code
        ORDER BY nh.scheme_code, nh.date
    """, conn, parse_dates=['date'])

fig, ax = plt.subplots(figsize=(14, 6))
for code in nav_all['scheme_code'].unique()[:8]:
    fund = nav_all[nav_all['scheme_code']==code].sort_values('date').set_index('date')
    rolling_cagr = fund['nav'].pct_change(252) * 100
    ax.plot(rolling_cagr.index, rolling_cagr.values,
            label=fund['scheme_name'].iloc[0][:22], lw=1, alpha=0.8)

ax.axhline(0,    color='black', lw=1,   ls='--')
ax.axhline(10,   color='green', lw=0.7, ls=':', alpha=0.6, label='10% threshold')
ax.axhline(-10,  color='red',   lw=0.7, ls=':', alpha=0.6)
ax.set_title('Rolling 1-Year CAGR (252-day window)', fontweight='bold')
ax.set_ylabel('Rolling 1Y CAGR %')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(ANA_DIR / 'adv_06_rolling_cagr.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("=" * 65)
print("ADVANCED ANALYTICS COMPLETE")
print("=" * 65)
charts = [
    ('adv_01_var.png',              'Value at Risk (Historical + Parametric)'),
    ('adv_02_monte_carlo.png',      'Monte Carlo Simulation (1Y projection)'),
    ('adv_03_efficient_frontier.png','Markowitz Efficient Frontier'),
    ('adv_04_clustering.png',       'Fund Clustering (K-Means, k=4)'),
    ('adv_06_rolling_cagr.png',     'Rolling CAGR Consistency'),
]
outputs = [
    ('var_analysis.csv',            'VaR & CVaR for all equity funds'),
    ('portfolio_weights.csv',       'Max Sharpe & Min Vol portfolio weights'),
    ('fund_clusters.csv',           'Fund cluster assignments'),
]
print("\nCHARTS:")
for f, desc in charts:
    print(f"  {f:40s} ← {desc}")
print("\nCSV EXPORTS:")
for f, desc in outputs:
    print(f"  {f:40s} ← {desc}")
print()
print("  ✅ Proceed to D5 (Power BI Dashboard) or D7 (Final Report)")
